In [ ]:
import numpy as np
import math

In [ ]:
class Calculate_Returns_Simple():
    def __init__(self, data, fee, initial_cash=10000):
        self.data = data
        self.fee = fee
        self.initial_cash = initial_cash
        self.returns = []
        self.records = []
        self.context = 0
        
    def sharpe(self, period = 365):
        returns_array = np.array([float(r) for r in self.returns])
        if returns_array.std() == 0:
            return 0
        return (returns_array.mean() / returns_array.std()) * math.sqrt(period)
    
    def equity_curve(self):
        returns_array = np.array([float(r) for r in self.returns])
        equity = np.cumprod(1 + returns_array) * float(self.initial_cash)
        return equity
    
    def cumulative_return(self):
        returns_array = np.array([float(r) for r in self.returns])
        if len(returns_array) == 0:
            return 0
        return np.prod(1 + returns_array) - 1
    
    def max_drawdown(self):
        equity = self.equity_curve()
        if len(equity) == 0:
            return 0
        
        running_max = np.maximum.accumulate(equity)
        drawdowns = (equity - running_max) / running_max
        return drawdowns.min()
    
    def annualized_return(self, period=365):
        cum_ret = self.cumulative_return()
        n_periods = len(self.returns)
        
        if n_periods == 0:
            return 0
        
        return (1 + cum_ret) ** (period / n_periods) - 1
    
    def calmar_ratio(self):
        mdd = abs(self.max_drawdown())
        if mdd == 0:
            return 0
        return self.annualized_return() / mdd
    
    def sortino(self, period=365):
        returns_array = np.array([float(r) for r in self.returns])
        downside = returns_array[returns_array < 0]
        
        if len(downside) == 0:
            return 0
        
        downside_std = downside.std()
        if downside_std == 0:
            return 0
        
        return (returns_array.mean() / downside_std) * math.sqrt(period)
    
    def win_rate(self):
        if len(self.records) == 0:
            return 0
        
        wins = sum(1 for trade in self.records if trade["PnL"] > 0)
        return wins / len(self.records)
    
    def profit_factor(self):
        profits = sum(trade["PnL"] for trade in self.records if trade["PnL"] > 0)
        losses = abs(sum(trade["PnL"] for trade in self.records if trade["PnL"] < 0))
        
        if losses == 0:
            return float("inf")
        
        return float(profits / losses)
        
    def calc_returns(self, currPrice, prevPrice):
        return (currPrice - prevPrice) / prevPrice
    
    def reset(self):
        self.returns = []
        self.records = []
        self.context = 0 # 0: hold, 1: short, 2: long
    
    def append_context_and_unrealized_pnl_to_state(self, state, index):
        if not self.records or self.context == 0:
            unrealized_pnl = 0
        else:
            unrealized_pnl = self.calc_returns(self.data[index], self.records[-1]['price'])
            unrealized_pnl = unrealized_pnl * -1 if self.context == 1 else unrealized_pnl
        unrealized_pnl = np.clip(unrealized_pnl, -1, 1)
        position_one_hot = np.zeros(3)
        position_index = int(self.context)
        position_one_hot[position_index] = 1
        return np.concatenate([state, position_one_hot, [unrealized_pnl]])
    
    def close_last_position(self, last_price):
        if not self.records:
            return
        last = self.records[-1]
        if 'PnL' not in last:
            if last['action'] == 2:  # long
                pnl = last_price - last['price']
            else:  # short
                pnl = last['price'] - last_price
            last['PnL'] = pnl
    
    def perform_action(self, action, index):
        if action == 0: #Hold
            if self.context == 0: #Hold while holding
                ret = 0
                self.returns.append(ret)
            elif self.context == 1: #Hold while short => continue short
                ret = -self.calc_returns(self.data[index], self.data[index-1])
                self.returns.append(ret)
            elif self.context == 2: #Hold while long => continue long
                ret = self.calc_returns(self.data[index], self.data[index-1])
                self.returns.append(ret)
        elif action == 1: #Short
            if self.context == 0: #Short while hold => open short
                self.context = 1
                ret = -self.calc_returns(self.data[index], self.data[index-1])
                ret -= self.fee
                self.returns.append(ret)
                self.records.append({
                    'action': 1,
                    'price': self.data[index],
                    'open_index': index,
                })
            elif self.context == 1: # Short while short => continue short
                ret = -self.calc_returns(self.data[index], self.data[index-1])
                self.returns.append(ret)
            elif self.context == 2: # Short while long => close long and open short
                self.context = 1
                ret = self.calc_returns(self.data[index], self.data[index-1])
                ret -= self.fee * 2
                self.returns.append(ret)
                self.records[-1]['PnL'] = self.data[index] - self.records[-1]['price'] - self.fee * self.data[index] - self.fee * self.records[-1]['price']
                self.records.append({
                    'action': 1,
                    'price': self.data[index],
                    'open_index': index,
                })
        elif action == 2: #Long
            if self.context == 0: #Long while hold => open long
                self.context = 2
                ret = self.calc_returns(self.data[index], self.data[index-1])
                ret -= self.fee
                self.returns.append(ret)
                self.records.append({
                    'action': 2,
                    'price': self.data[index],
                    'open_index': index,
                })
            elif self.context == 1: #Long while short => close short and open long
                self.context = 2
                ret = -self.calc_returns(self.data[index], self.data[index-1])
                ret -= self.fee * 2
                self.returns.append(ret)
                self.records[-1]['PnL'] = self.records[-1]['price'] - self.data[index] - self.fee * self.records[-1]['price'] - self.fee * self.data[index] 
                self.records.append({
                    'action': 2,
                    'price': self.data[index],
                    'open_index': index,
                })
            elif self.context == 2: # Long while long => continue long
                ret = self.calc_returns(self.data[index], self.data[index-1])
                self.returns.append(ret)
        return ret
            